**生成式AI的使用 (Generative AI Use)**：就本次作业而言，生成式AI的使用同样受关于协作的政策约束。与其他合作者一样，每个学生必须独立于（与AI）交互的输出写下解答，并且在提交的作业中应包含说明协作性质的附注。使用生成式AI工具来实质性地完成作业的大部分内容不符合本次作业的精神，这一行为将违反[荣誉守则](https://communitystandards.stanford.edu/policies-and-guidance/honor-code)。

In [ ]:
# 将你的 Google Drive 挂载到 Colab 虚拟机上。
from google.colab import drive
drive.mount('/content/drive')

# TODO: 输入你在 Drive 中保存解压后作业文件夹的文件夹名称，
# 例如 'cs231n/assignments/assignment2/'
FOLDERNAME = 'cs231n/assignments/assignment2/'
assert FOLDERNAME is not None, "[!] 输入文件夹名称。([!] Enter the foldername.)"

# 现在我们已经挂载了你的 Drive，这确保了
# Colab 虚拟机的 Python 解释器能够从中加载 python 文件。
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

# 如果你的 Drive 中尚未存在 CIFAR-10 数据集，
# 以下代码会将其下载到你的 Drive 中。
%cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/datasets/
!bash get_datasets.sh
%cd /content/drive/My\ Drive/$FOLDERNAME

# Dropout (随机失活)
Dropout [1] 是一种用于正则化神经网络的技术，通过在正向传播的过程中随机将一些输出的激活设置为零。在本次练习中，你将实现一个 Dropout 层并对你的全连接网络结构做修改以便去有选择地使用 Dropout 功能。

[1] [Geoffrey E. Hinton et al, "Improving neural networks by preventing co-adaptation of feature detectors", arXiv 2012](https://arxiv.org/abs/1207.0580)

In [ ]:
# 设置单元格。
import time
import numpy as np
import matplotlib.pyplot as plt
from cs231n.classifiers.fc_net import *
from cs231n.data_utils import get_CIFAR10_data
from cs231n.gradient_check import eval_numerical_gradient, eval_numerical_gradient_array
from cs231n.solver import Solver

%matplotlib inline
plt.rcParams["figure.figsize"] = (10.0, 8.0)  # 设置绘图的默认大小。
plt.rcParams["font.family"] = "SimHei"  # 设置字体以支持中文显示
plt.rcParams["axes.unicode_minus"] = False  # 解决负号 '-' 显示为方块的问题
plt.rcParams["image.interpolation"] = "nearest"
plt.rcParams["image.cmap"] = "gray"

import sys
import types
import importlib

if "imp" not in sys.modules:
    imp = types.ModuleType("imp")
    imp.reload = importlib.reload
    sys.modules["imp"] = imp

%load_ext autoreload
%autoreload 2

def rel_error(x, y):
    """返回相对误差 (Returns relative error.)。"""
    return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

In [ ]:
# 加载（经过预处理的）CIFAR-10 数据。
data = get_CIFAR10_data()
for k, v in list(data.items()):
    print(f"{k}: {v.shape}")

# Dropout: 前向传播
在 `cs231n/layers.py` 文件中，实现 Dropout 的前向传播。考虑到 Dropout 在训练以及测试时拥有这完全不一样的功能表现，所以确保你对该两种模式下的操作运算进行了各自的程序实现。

一旦你完成了这件事情，运行后面的这些单元格来对你的所写的进行一个完整的运作测试。

In [ ]:
np.random.seed(231)
x = np.random.randn(500, 500) + 10

for p in [0.25, 0.4, 0.7]:
    out, _ = dropout_forward(x, {'mode': 'train', 'p': p})
    out_test, _ = dropout_forward(x, {'mode': 'test', 'p': p})

    print('带 p 运行检测中 (Running tests with p) = ', p)
    print('输入的均值 (Mean of input): ', x.mean())
    print('训练阶段输出的均值 (Mean of train-time output): ', out.mean())
    print('测试阶段输出的均值 (Mean of test-time output): ', out_test.mean())
    print('训练阶段输出归零元素的比例分数 (Fraction of train-time output set to zero): ', (out == 0).mean())
    print('测试阶段输出归零元素的比例分数 (Fraction of test-time output set to zero): ', (out_test == 0).mean())
    print()

# Dropout：反向传播
在 `cs231n/layers.py` 文件中寻找并实现针对这个 Dropout 函数构建的的反向传播过程。在你执行完成以后，跑接下来呈现出来的这些相关方框里包含的一系列算法脚本给你的代码部分所涉及数值导回做相关的核验。

In [ ]:
np.random.seed(231)
x = np.random.randn(10, 10) + 10
dout = np.random.randn(*x.shape)

dropout_param = {'mode': 'train', 'p': 0.2, 'seed': 123}
out, cache = dropout_forward(x, dropout_param)
dx = dropout_backward(dout, cache)
dx_num = eval_numerical_gradient_array(lambda xx: dropout_forward(xx, dropout_param)[0], x, dout)

# 误差应在 e-10 或更小的量级左右 (Error should be around e-10 or less.)
print('dx 相对误差 (dx relative error): ', rel_error(dx, dx_num))

## 内联问题 1 (Inline Question 1)：
如果我们在 Dropout 层里没有将在应用反向 dropout 传出来的值给去除以那个 `p` 会发生什么事？然后为什么又会导致那样的事被引发呢？

## 解答：
[在此处写入补充作答的内容]



# 含 Dropout 的全连接网络
打开对应的此个位置目录的文书即在这里：`cs231n/classifiers/fc_net.py`，去将你们本身这边的操作方式中添加入 Dropout 这项用法。其详情大致指的是这样的，哪怕整个对于这种构架的设计的该构建部分获得用于所给到的 `dropout_keep_ratio` 数值上的这一个输入并不属于这个默认情况原本是的一的数字的这个样子时候，我们设计网络也就得要去在这每段的给执行完 ReLU 这步骤之后添加补回上此这特有相关的一类 Dropout 图层了。完毕所上说的内容以后，可以执行下面的这里来获取所对编写完成对应内容的算法部分开展那关于这个数据方面对应的各项梯度验证审查了。

In [ ]:
np.random.seed(231)
N, D, H1, H2, C = 2, 15, 20, 30, 10
X = np.random.randn(N, D)
y = np.random.randint(C, size=(N,))

for dropout_keep_ratio in [1, 0.75, 0.5]:
    print('执行 dropout 参数检验 (Running check with dropout) = ', dropout_keep_ratio)
    model = FullyConnectedNet(
        [H1, H2],
        input_dim=D,
        num_classes=C,
        weight_scale=5e-2,
        dtype=np.float64,
        dropout_keep_ratio=dropout_keep_ratio,
        seed=123
    )

    loss, grads = model.loss(X, y)
    print('初始阶段给出的损失 (Initial loss): ', loss)

    # 给这里计算得到的这个相对的各类偏差度差不多应是在属于这关于它的那类似达到 e-6 或再怎么可能少一些范围之类的 (Relative errors should be around e-6 or less.)
    # 并且值得你得留意到在设置的这一类关于把 dropout_keep_ratio 指向到了是1 那个样子这时候你去拿在这期间拥有的关于它所涉及到了此W2误差情况上到达这种处于像它这般差不多位于类似于e-5那等同之下的状态，其实是一点问题都不妨碍能够算是正确的。(Note that it's fine if for dropout_keep_ratio=1 you have W2 error be on the order of e-5.)
    for name in sorted(grads):
        f = lambda _: model.loss(X, y)[0]
        grad_num = eval_numerical_gradient(f, model.params[name], verbose=False, h=1e-5)
        print('%s 的相对差错等价结果 (relative error): %.2e' % (name, rel_error(grad_num, grads[name])))
    print()

# 正则化实验 (Regularization Experiment)
在实验的一步骤里，我们将用 500 个相关类型的这部分的给的用来受训样本给一堆的两层式神经网络模型们跑起相关这类受其指引的内容训练，其中里面有的这一半，那部分的是对于本身一点 dropout 也没有用的那一套，和别的一种，此即属于保留比上用成拥有了那 0.25 此设数值的那套设计；随着后续相关步骤在跟上了我们之后就会把这俩等各自网架通过对于相关这类涉及受这一时间影响的过程当中能看到所获得对应的表现准确类评估所拥有的给出来的有关受其自身这里面验证数据上这些种种状况展现拿来予以把呈现以一类有形的模式做展现了。

In [ ]:
# 去执行有关所讲这些俩有着同样设的对应这种模型，其中的这一个是含有这所附加进去了有的这属于dropout部分的另一类则是无任何含有之上的。(Train two identical nets, one with dropout and one without.)
np.random.seed(231)
num_train = 500
small_data = {
    'X_train': data['X_train'][:num_train],
    'y_train': data['y_train'][:num_train],
    'X_val': data['X_val'],
    'y_val': data['y_val'],
}

solvers = {}
dropout_choices = [1, 0.25]
for dropout_keep_ratio in dropout_choices:
    model = FullyConnectedNet(
        [500],
        dropout_keep_ratio=dropout_keep_ratio
    )
    print(dropout_keep_ratio)

    solver = Solver(
        model,
        small_data,
        num_epochs=25,
        batch_size=100,
        update_rule='adam',
        optim_config={'learning_rate': 5e-4,},
        verbose=True,
        print_every=100
    )
    solver.train()
    solvers[dropout_keep_ratio] = solver
    print()

In [ ]:
# 去绘画所属于这些关于对于模型它们这在训练环节与用于它们被给作做属于此相关之类验证中取得到了对应的有关这里准许数据的信息线图。 (Plot train and validation accuracies of the two models.)
train_accs = []
val_accs = []
for dropout_keep_ratio in dropout_choices:
    solver = solvers[dropout_keep_ratio]
    train_accs.append(solver.train_acc_history[-1])
    val_accs.append(solver.val_acc_history[-1])

plt.subplot(3, 1, 1)
for dropout_keep_ratio in dropout_choices:
    plt.plot(
        solvers[dropout_keep_ratio].train_acc_history, 'o', label='%.2f 对应这设里的部分给它保留比 (dropout_keep_ratio)' % dropout_keep_ratio)
plt.title('关于在此期间对于对应上于这里被运用而能获所得出来涉及的属于在此的培训期间精确之率 (Train accuracy)')
plt.xlabel('代 (Epoch)')
plt.ylabel('准确度 (Accuracy)')
plt.legend(ncol=2, loc='lower right')

plt.subplot(3, 1, 2)
for dropout_keep_ratio in dropout_choices:
    plt.plot(
        solvers[dropout_keep_ratio].val_acc_history, 'o', label='%.2f 对应这设里的部分给它保留比 (dropout_keep_ratio)' % dropout_keep_ratio)
plt.title('去验在它们身上相关的表现精确等之类对应准了的信息数图等 (Val accuracy)')
plt.xlabel('代 (Epoch)')
plt.ylabel('准确度 (Accuracy)')
plt.legend(ncol=2, loc='lower right')

plt.gcf().set_size_inches(15, 15)
plt.show()

## 内联问题 2 (Inline Question 2)：
把那个跟这里所去有和不拥有的属于那叫做 dropout 配置情况的这个有相关的这些属于这里的它们身上被开展而做的验正上的效果和受到做那方面受的那些等准等拿去做属于在这点上的一些比对比—从这种结果上看来说明了这个作为这里等算起在这之上被利用在去关于在对应这属于类似这规范相关之类的方面是能说明在这当头上这 dropout 上可以带来怎样的这一影响表现内容呢？

## 解答：
[填写这这里空缺的信息以予作答在此上]

